# `07_RAG1.ipynb`

```sh
uv add "langchain[openai]" langchain-text-splitters requests numpy deepagents
```

## Store (저장)

In [ ]:
from dotenv import load_dotenv
load_dotenv()

DOCS_BASE = "https://docs.langchain.com"

# Curated LangChain OSS pages for this tutorial. Expand this list or parse
# URLs from https://docs.langchain.com/llms.txt to index more of the site.
DOC_PATHS = [
    "/oss/python/langchain/agents",
    "/oss/python/deepagents/rag",
    "/oss/python/langchain/tools",
    "/oss/python/langchain/models",
    "/oss/python/deepagents/retrieval",
    "/oss/python/langchain/knowledge-base",
    "/oss/python/langchain/middleware",
    "/oss/python/deepagents/overview",
    "/oss/python/deepagents/subagents",
    "/oss/python/deepagents/streaming",
    "/oss/python/deepagents/frontend/subagent-streaming",
    "/oss/python/deepagents/backends",
    "/oss/python/langgraph/overview",
    "/oss/python/langgraph/quickstart",
]

In [ ]:
# 1. Load (PDF, HTML, TEXT, MD, IMG, VIDEO, HWPX, XLSX, PPTX, DOCX) -> 문서 종류에 따라 방법이 다름

import requests
from langchain_core.documents import Document  # RAG에 사용할 문서 쪼가리를 의미하는 데이터 타입

# Node, Edge 이런거 아님. 단순 함수
def load_langchain_docs():
    docs = []

    for path in DOC_PATHS:
        url = f'{DOCS_BASE}{path}.md'
        res = requests.get(url, timeout=5)  # 5초간 답이 없으면 넘어가라
        # 단순 str 말고 Document 타입으로 잘 감싸기 -> RAG에 사용하기 위해
        doc = Document(page_content=res.text, metadata={'source': f'{DOCS_BASE}{path}'})
        docs.append(doc)

    return docs

docs = load_langchain_docs()
print(f'{len(docs)}개의 문서를 불러왔습니다')

In [ ]:
# 2. Split
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

splits = splitter.split_documents(docs)
print(f'{len(splits)}개의 조각으로 잘랐습니다')

In [ ]:
# 3. Embed
from langchain_openai import OpenAIEmbeddings

# embedding 담당자
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

In [ ]:
# 4. Store
from langchain_core.vectorstores import InMemoryVectorStore

# 벡터스토어 세팅(임베딩)
vectorstore = InMemoryVectorStore(embedding=embeddings)

# 벡터스토어 저장 (문서조각)
vectorstore.add_documents(documents=splits)  # 리턴값은 저장된 문서 ID (쓸모없음)
print('저장완료')

In [22]:
answer = ''

docs = vectorstore.similarity_search('langchain basic')

print('\n---\n'.join(map(lambda doc: doc.page_content, docs)))

<Expandable title="how LangChain products fit together" defaultOpen={false}>
  * [Deep Agents](/oss/python/deepagents/overview) is an [agent harness](/oss/python/concepts/products#agent-harnesses-like-the-deep-agents-sdk): planning, subagents, filesystem tools, and context management on top of LangGraph.
  * [LangChain](/oss/python/langchain/overview) is the agent framework: abstractions and integrations for models, tools, and agent loops.
  * [LangGraph](/oss/python/langgraph/overview) is the orchestration runtime: durable execution, streaming, human-in-the-loop, and persistence.
  * [LangSmith](/langsmith/observability) is the platform for tracing, evaluation, prompts, and deployment across frameworks.
  * [LangSmith Engine](/langsmith/engine) detects issues in your LangGraph agent traces and proposes fixes. You can open a pull request with the proposed fix directly from the Engine tab.
---
<Card title="LangChain" icon="https://mintcdn.com/langchain-5e9cc07a/nQm-sjd_MByLhgeW/images/b

## Retrieve (검색)

In [26]:
# 1. Tool 만들기
from langchain.tools import tool


# parse_doctsting=True -> docstirng 을 google style로 잘 작성했다면, Agent가 더 잘 이해함
@tool(parse_docstring=True)
def rag_search_document(query: str):
    """Search LangChain documentation.

    Args:
        query: Natural language search query.
    """
    retrieved_docs = vectorstore.similarity_search(query, k=4)  # 문서 4개 검색

    # 커다란 문자열로 합쳐서
    result = '\n---\n'.join(map(lambda doc: doc.page_content, retrieved_docs))
    return result
    

In [ ]:
from langchain.agents import create_agent
# 2. Agent 만들어서 Tool 쥐어주기
agent = create_agent(
    model='openai:gpt-4.1-mini',
    tools=[rag_search_document],
    system_prompt='''사용자 질문에 답하는 에이전트
- 사용자가 Langchain 이나 LLM 관련 질문을 하면
rag_serach_document 도구를 사용하여 답변
'''
)

# agent -> graph -> state -> 'messages' state -> invoke 할때는 state를 넣어줌 -> 끝날때 state
result = agent.invoke({
    'messages': [
        {'role': 'user', 'content': 'langchain 에서 RAG를 하는법을 알려줘'}
    ]
})

In [34]:
for msg in result['messages']:
    msg.pretty_print()

================================ Human Message =================================

langchain 에서 RAG를 하는법을 알려줘
================================== Ai Message ==================================
Tool Calls:
  rag_search_document (call_VMzB4rV9URRggT6A6QNm0ZIz)
 Call ID: call_VMzB4rV9URRggT6A6QNm0ZIz
  Args:
    query: langchain RAG 방법
================================= Tool Message =================================
Name: rag_search_document

For more on RAG:

* [Retrieval overview](/oss/python/deepagents/retrieval)
* [RAG with Deep Agents](/oss/python/deepagents/rag)
* [Evaluate a RAG application](/langsmith/evaluate-rag-tutorial)

***

<div className="source-links">
  <Callout icon="terminal-2">
    [Connect these docs](/use-these-docs) to Claude, VSCode, and more via MCP for real-time answers.
  </Callout>

  <Callout icon="edit">
    [Edit this page on GitHub](https://github.com/langchain-ai/docs/edit/main/src/oss/langchain/knowledge-base.mdx) or [file an issue](https://github.com/langcha